In [12]:
import csv
import math

In [6]:
def mean(num_list):
    total = 0

    if not list:
        raise ValueError('Enter a non empty list')
    for item in num_list:
        total += item
    return total / len(num_list)


In [7]:
def median(num_list):
    if not num_list:
        raise ValueError('Enter a non empty list')
    sorted_list = sorted(num_list)
    list_length = len(sorted_list)

    middle = list_length // 2

    if list_length % 2 != 0:
        return sorted_list[middle]
    return (sorted_list[middle - 1] + sorted_list[middle]) / 2

In [8]:
def mode(num_list):
    if not num_list:
        raise ValueError('Enter a non empty list')

    counts = {}
    for item in num_list:
        counts[item] = counts.get(item, 0) + 1

    highest_frequency = max(counts.values())

    if highest_frequency == 1:
        return []
    return [item for item, frequency in counts.items() if frequency == highest_frequency]


In [10]:
def variance(values, sample):
    if not values:
            raise ValueError('Enter a non empty list')

    values_size = len(values)

    mean = sum(values) / values_size
    x_bar = [(value - mean)**2 for value in values]

    denominator = values_size - 1 if sample else values_size

    return sum(x_bar) / denominator

In [11]:
def std_dev(values, sample):
    return variance(values, sample) ** 0.5

In [26]:
def load_column(path, column):
    valid_floats = []
    rejected_list = []

    missing_values = ['', 'na', 'n/a', 'null', 'none', 'nan', None]

    with open(path, 'r') as f:
        reader = csv.DictReader(f)

        if not reader.fieldnames or column not in reader.fieldnames:
            raise ValueError(f'Column {column} not found in the file')

        for row_num, row in enumerate(reader):
            row_value = row.get(column)

            if row_value.strip().lower() in missing_values:
                reason = 'Missing Value'

            else:
                try:
                    value = float(row_value.strip())
                except (ValueError):
                    reason = 'Invalid value'
                else:
                    valid_floats.append(value)
                    continue
            rejected_list.append((row_num, row_value))
        return valid_floats, rejected_list


In [27]:
valid_floats, rejected_list = load_column('./data/agent_deposits.csv', 'daily_deposits_kes')
print(rejected_list)
print(valid_floats)

[(5, ''), (8, '-'), (12, 'n/a'), (23, '12,40O'), (27, '')]
[14582.97, 19708.28, 21684.0, 16857.73, 14247.27, 20029.64, 22014.19, 14362.83, 20758.97, 20017.07, 21643.87, 18174.3, 17622.44, 26725.69, 17575.83, 15988.21, 96500.0, 18805.47, 18965.0, 17396.43, 19373.89, 20715.67, 22009.77, 12772.7, 20745.52, 13956.97, 1200.0, 13783.85, 18506.72, 16008.59, 88400.0, 29060.99, 18344.46, 21195.44, 14265.36]


In [28]:
def quartiles(values):
    sorted_values = sorted(values)

    values_size = len(sorted_values)

    mid = values_size // 2
    first_half = sorted_values[:mid]
    last_half = sorted_values[(mid+1 if values_size % 2 else mid):]

    q1 = median(first_half)
    q2 = median(sorted_values)
    q3 = median(last_half)

    return q1, q2, q3


In [29]:
def iqr_outliers(values, k):
    q1, q2, q3 = quartiles(values)
    iqr = q3 - q1

    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr

    outliers = [
        value for value in values if value < lower_bound or value > upper_bound
    ]

    return {
        'IQR': iqr,
        'minimum_value': lower_bound,
        'maximum_value': upper_bound,
        'outliers': outliers
    }

In [ ]:
class StatisticsAnalyzer:
    def __init__(self, values, sample, k):
        self.values = values
        self.sample = sample
        self.k = k
        self.sorted_values = sorted(values)

@classmethod
def from_csv(cls, path, col, k, sample=False):
    values, rejected = load_column(path, col)

    analyzer = cls(values, sample=sample, k=k)
    analyzer.rejected = rejected

    return analyzer